In [1]:
#02.Buils our Spam detector
#step 1- read and preprocess data
#step 2- NLP techniques
#step 3- Tokenization
#step 4- Tokens to vectors--- for TD IDF token generate
#step 5- Naive bayes ( Multinominal NB for -Better Spam detection--> Precision score used  as a evalution metrixs)
#step 6- Logistics regression ( Project Condition so applied --> F1_score used for better in spam detection)\
#setp 7- Final sentiment polarity score is output for project(*condition(g))

In [2]:
from pydoc import text

import pandas as pd
import nltk, re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk import pos_tag, ne_chunk
from textblob import TextBlob
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, f1_score, recall_score,accuracy_score

#1st one for Multinominal NB,  2nd one for Logistic Regression
from textblob.en import sentiment


ModuleNotFoundError: No module named 'nltk'

In [ ]:
# --- 1. Load Data ---
data = pd.read_csv('spam 5.csv', encoding='latin-1')
data = data[['v1', 'v2']] # v1 is label, v2 is text

# --- 2. Label Encoding (y = Target) ---
le = LabelEncoder()
y = le.fit_transform(data['v1']) # Target: Spam or Ham


In [ ]:
# --- 3. NLP Pipeline (Processes v2 individually) ---

def vec_tokens(text):
    text=str(text).lower()
    clean_text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(clean_text)

    stop_words=set(stopwords.words('english'))
    tokens= [w for w in tokens if w not in stop_words ]

    lemmatizer= WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)

# Apply the pipeline to v2 ( the Content )
vec_tokens_result= data['v2'].apply(vec_tokens)

X_processed = data['v2'].apply(vec_tokens)

vectorizer = TfidfVectorizer()
X_vec= vectorizer.fit_transform(X_processed).toarray()

X_vec_train, X_vec_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

# skipped the stempping process because of the dataset { v2 columns - consist of lot of speacial character and un meaningfull words in that, so stemped take long time to process }

In [ ]:
# --- 6. Naive Bayes MultinomialNB ---
mnb = MultinomialNB()
mnb.fit(X_vec_train, y_train)
y_pred_mnb = mnb.predict(X_vec_test)

accuracy=accuracy_score(y_test, y_pred_mnb)
print(f'Accuracy Score: {accuracy:.2f}%')
Precision = precision_score(y_test, y_pred_mnb)
print(f'Precision: {Precision * 100:.2f}%')
recall=recall_score(y_test, y_pred_mnb)
print(f'Recall: {recall * 100:.2f}%')
F1=f1_score(y_test, y_pred_mnb)
print(f'F1: {F1 * 100:.2f}%')

In [ ]:
# --- 7. Logistic Regression ---
lr = LogisticRegression()
lr.fit(X_vec_train, y_train)
y_pred_lr = lr.predict(X_vec_test)

accuracy=accuracy_score(y_test, y_pred_lr)
print(f'Accuracy Score: {accuracy:.2f}%')
precision=precision_score(y_test, y_pred_lr)
print(f'Precision: {precision * 100:.2f}%')
recall=recall_score(y_test, y_pred_lr)
print(f'Recall: {recall * 100:.2f}%')
F1=f1_score(y_test, y_pred_lr)
print(f'F1 score: {F1 * 100:.2f}%')

In [ ]:
def setiment_analysis(text):
    text=str(text).lower()
    clean_text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(clean_text)

  # stemmer = PorterStemmer()
  # tokens = [stemmer.stem(w) for w in tokens]
  # lemmatizer = WordNetLemmatizer()
  # tokens=[lemmatizer.lemmatize(w) for w in tokens]

    final_text= ' '.join(tokens)
    sentiment= TextBlob(final_text).sentiment.polarity

    return {
        'sentiment score by polarity': sentiment,
        'polarity': '+ve' if sentiment > 0 else '-ve' if sentiment < 0 else 'neutral'
    }
texts = data['v2']

for i, text in enumerate(texts, start=1):
    result = setiment_analysis(text)
    print(f"\nSentence {i}: {text}")
    print(f"Sentiment Score by Polarity: {result['sentiment score by polarity']:.2f}%")
    print(f"Polarity: {result['polarity']}")
